# Create OEE Manufacturing Ontology


This notebook creates a Fabric ontology for the OEE Manufacturing Demo.


## Architecture
- **6 entity types**: 3 dimensions + 3 raw events
- **8 relationships**: Connecting events to lines/stations/schedules
- **Hybrid binding strategy**:
  - Static bindings (indexed in graph): Lakehouse via OneLake availability
  - Time series bindings (real-time lookup): Eventhouse KQL tables


**Note**: OEEMetric (aggregated from materialized view) is excluded because KQL materialized views don't replicate to OneLake and therefore can't have NonTimeSeries bindings.


## Prerequisites
1. Fabric workspace with Real-Time Intelligence enabled
2. ManufacturingEH Eventhouse with KQL database and tables created
3. OneLake availability enabled on the KQL database (Settings → OneLake availability)
4. Ontology CSV files uploaded to lakehouse Files/ontology/ folder
5. Download the latest `fabriciq_ontology_accelerator` wheel from https://github.com/microsoft/fabriciq-accelerator/releases and upload it to lakehouse Files/ontology/ before running Step 1

## Step 1: Install Fabric IQ Ontology Accelerator

In [ ]:
%pip install /lakehouse/default/Files/ontology/fabriciq_ontology_accelerator-0.1.0-py3-none-any.whl --q

## Step 2: Import Packages & Get Workspace Context

In [ ]:
import sempy.fabric as fabric
from fabricontology import generate_definition_from_package, create_ontology_item
import notebookutils
import os
import zipfile

In [ ]:
# Get workspace ID
workspace_id = fabric.get_workspace_id()
print(f"✓ Workspace ID: {workspace_id}")

# List all items to find Eventhouse and Lakehouse
items_df = fabric.list_items()

# Get Eventhouse details
eventhouse_name = "ManufacturingEH"
eventhouse_items = items_df[(items_df["Type"] == "Eventhouse") & (items_df["Display Name"] == eventhouse_name)]
if eventhouse_items.empty:
    raise ValueError(f"Eventhouse '{eventhouse_name}' not found in workspace")

eventhouse_id = str(eventhouse_items.iloc[0].Id)
print(f"✓ Eventhouse ID: {eventhouse_id}")

# Get KQL Database details to extract cluster URI
kql_db_items = items_df[(items_df["Type"] == "KQLDatabase") & (items_df["Display Name"] == eventhouse_name)]
if kql_db_items.empty:
    raise ValueError(f"KQL Database '{eventhouse_name}' not found in workspace")

kql_db_id = str(kql_db_items.iloc[0].Id)

# Get cluster URI from KQL Database using REST API
import requests
access_token = notebookutils.credentials.getToken("pbi")
headers = {
    "Authorization": f"Bearer {access_token}",
    "Accept": "application/json"
}

kql_db_resp = requests.get(
    f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/kqlDatabases/{kql_db_id}",
    headers=headers
)

if kql_db_resp.status_code == 200:
    kql_db_detail = kql_db_resp.json()
    eventhouse_uri = kql_db_detail["properties"]["queryServiceUri"]
    print(f"✓ Eventhouse URI: {eventhouse_uri}")
else:
    raise ValueError(f"Failed to get KQL Database details: {kql_db_resp.status_code} - {kql_db_resp.text}")

# Get Lakehouse for OneLake bindings (KQL tables accessed via OneLake shortcuts)
lakehouse_name = "ManufacturingLH"
lakehouse_items = items_df[(items_df["Type"] == "Lakehouse") & (items_df["Display Name"] == lakehouse_name)]
if lakehouse_items.empty:
    raise ValueError(f"Lakehouse '{lakehouse_name}' not found in workspace")

lakehouse_id = str(lakehouse_items.iloc[0].Id)
print(f"✓ Lakehouse ID: {lakehouse_id}")

# Ontology name
ontology_name = "OEEManufacturingOntology"
print(f"✓ Ontology name: {ontology_name}")

## Step 3: Load & Process CSV Files

In [ ]:
# Path to ontology CSV files in lakehouse Files section
ontology_base_path = "/lakehouse/default/Files/ontology"

# CSV files to include
csv_files = [
    ("definition/entity_types.csv", "Entity type definitions"),
    ("definition/relationship_types.csv", "Relationship type definitions"),
    ("binding/binding_entity_types.csv", "Property bindings"),
    ("binding/binding_relationship_types.csv", "Relationship bindings")
]

# Verify all files exist
print("Verifying CSV files in lakehouse Files/ontology/...")
missing_files = []
for file_path, description in csv_files:
    full_path = f"{ontology_base_path}/{file_path}"
    if os.path.exists(full_path):
        file_size = os.path.getsize(full_path) / 1024
        print(f"  ✓ {file_path} ({file_size:.1f} KB)")
    else:
        print(f"  ❌ {file_path} (NOT FOUND)")
        missing_files.append(file_path)

if missing_files:
    raise FileNotFoundError(f"Required CSV files not found: {missing_files}")

## Step 3b: Validate CSVs (catch silent failures before creation)

Based on lessons from FabricOracleHFMDemo ontology deployment:
1. **NonTimeSeries bindings must only reference static properties** (IsTimeseries=FALSE)
2. **TargetKeyColumnNames must reference columns in the SOURCE table** (not the target)
3. **Property data types must match actual table column types**

In [ ]:
import csv

def load_csv(path):
    with open(path, 'r') as f:
        return list(csv.DictReader(f))

entity_rows = load_csv(f"{ontology_base_path}/definition/entity_types.csv")
binding_rows = load_csv(f"{ontology_base_path}/binding/binding_entity_types.csv")
rel_bind_rows = load_csv(f"{ontology_base_path}/binding/binding_relationship_types.csv")

errors = []

# --- Check 1: NonTimeSeries bindings must NOT reference timeseries properties ---
for i, row in enumerate(binding_rows):
    if row["DataBindingType"] == "NonTimeSeries" and row["IsTimeseries"] == "TRUE":
        errors.append(
            f"CHECK 1 FAIL: binding_entity_types.csv row {i+2}: "
            f"NonTimeSeries binding for {row['EntityTypeName']}.{row['PropertyName']} "
            f"references a timeseries property (IsTimeseries=TRUE)"
        )

# --- Check 2: TargetKeyColumnNames must exist as columns in the SOURCE entity ---
entity_props = {(r["EntityTypeName"], r["PropertyName"]) for r in entity_rows}
for row in rel_bind_rows:
    src = row["SourceEntityTypeName"]
    for col in row["TargetKeyColumnNames"].split(";"):
        if (src, col) not in entity_props:
            errors.append(
                f"CHECK 2 FAIL: binding_relationship_types.csv: "
                f"{row['RelationshipName']} TargetKeyCol '{col}' not found in {src}"
            )

# --- Check 3: Relationship key columns must be static (not timeseries) ---
ts_map = {(r["EntityTypeName"], r["PropertyName"]): r["IsTimeseries"] for r in entity_rows}
for row in rel_bind_rows:
    src = row["SourceEntityTypeName"]
    for col in row["TargetKeyColumnNames"].split(";"):
        if ts_map.get((src, col)) == "TRUE":
            errors.append(
                f"CHECK 3 FAIL: binding_relationship_types.csv: "
                f"{row['RelationshipName']} TargetKeyCol '{col}' in {src} is timeseries"
            )

# --- Check 4: Entity identifiers must be String or BigInt ---
for row in entity_rows:
    if row["IsIdentifier"] == "TRUE" and row["PropertyDataType"] not in ("String", "BigInt"):
        errors.append(
            f"CHECK 4 FAIL: entity_types.csv: "
            f"{row['EntityTypeName']}.{row['PropertyName']} is identifier with type "
            f"{row['PropertyDataType']} (must be String or BigInt)"
        )

# --- Check 5: Binding types consistent with entity definition ---
entity_ts = {(r["EntityTypeName"], r["PropertyName"]): r["IsTimeseries"] for r in entity_rows}
for i, row in enumerate(binding_rows):
    key = (row["EntityTypeName"], row["PropertyName"])
    def_ts = entity_ts.get(key)
    bind_ts = row["IsTimeseries"]
    if def_ts is not None and def_ts != bind_ts:
        # The timeseries duplicate of identifier is expected to differ
        if row["DataBindingType"] == "TimeSeries" and def_ts == "FALSE":
            continue  # This is the identifier's TimeSeries duplicate row - OK
        errors.append(
            f"CHECK 5 FAIL: binding_entity_types.csv row {i+2}: "
            f"{row['EntityTypeName']}.{row['PropertyName']} IsTimeseries={bind_ts} "
            f"but entity_types.csv says {def_ts}"
        )

# --- Report ---
if errors:
    print(f"❌ {len(errors)} validation error(s) found:\n")
    for e in errors:
        print(f"  {e}")
    raise ValueError(f"CSV validation failed with {len(errors)} error(s). Fix the CSVs and re-upload before proceeding.")
else:
    print(f"✓ All validation checks passed:")
    print(f"  Entity types: {len(set(r['EntityTypeName'] for r in entity_rows))}")
    print(f"  Properties: {len(entity_rows)}")
    print(f"  Bindings: {len(binding_rows)} ({sum(1 for r in binding_rows if r['DataBindingType']=='NonTimeSeries')} static + {sum(1 for r in binding_rows if r['DataBindingType']=='TimeSeries')} timeseries)")
    print(f"  Relationships: {len(rel_bind_rows)}")
    print(f"  No NonTimeSeries→timeseries violations")
    print(f"  All TargetKeyColumnNames exist in source entities")
    print(f"  All identifiers are String or BigInt")

## Step 4: Create .iq Package

In [ ]:
# Create .iq package (ZIP file) from CSV files
import tempfile
temp_dir = tempfile.mkdtemp()
iq_package_path = os.path.join(temp_dir, "oee_manufacturing_ontology.iq")

print(f"\nCreating .iq package at: {iq_package_path}")

with zipfile.ZipFile(iq_package_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file_path, description in csv_files:
        full_path = f"{ontology_base_path}/{file_path}"
        zipf.write(full_path, file_path)
        print(f"  Added: {file_path}")

print(f"\n✓ Package created successfully ({os.path.getsize(iq_package_path) / 1024:.1f} KB)")

## Step 5: Generate Ontology Definition

In [ ]:
print("\nGenerating ontology definition from package...")

# The library will substitute placeholders in the CSV files based on SourceType:
# - LakehouseTable rows use binding_lakehouse_item_id
# - EventhouseTable rows use binding_eventhouse_item_id
result_tuple = generate_definition_from_package(
    ontology_package_path=iq_package_path,
    ontology_name=ontology_name,
    binding_workspace_id=workspace_id,
    binding_lakehouse_item_id=lakehouse_id,
    binding_lakehouse_schema_name="dbo",  # Schema for KQL tables via OneLake shortcuts
    binding_eventhouse_item_id=eventhouse_id,
    binding_eventhouse_cluster_uri=eventhouse_uri,
    binding_eventhouse_database_name="ManufacturingEH"
)

# Extract ontology definition (first element of the tuple)
ontology_definition = result_tuple[0]

print("✓ Definition generated successfully")
print(f"  Entity types: {len(result_tuple[1])}")
print(f"  Relationship types: {len(result_tuple[2])}")
print(f"  Data bindings: {len(result_tuple[3])}")

## Step 6: Create Ontology in Fabric

In [ ]:
# Get access token
access_token = notebookutils.credentials.getToken("pbi")
print("✓ Access token obtained")

In [ ]:
import requests

headers = {
    "Authorization": f"Bearer {access_token}",
    "Accept": "application/json",
    "Content-Type": "application/json",
}

# Check if an ontology with this name already exists
print(f"Checking for existing ontology '{ontology_name}'...")
list_resp = requests.get(
    f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items",
    headers=headers,
)

existing = [
    i for i in list_resp.json().get("value", [])
    if i.get("type") == "Ontology" and i.get("displayName") == ontology_name
]

if existing:
    ontology_id = existing[0]["id"]
    raise ValueError(
        f"❌ Ontology '{ontology_name}' already exists (id={ontology_id}).\n"
        f"   Please delete it manually from the workspace before creating a new one."
    )

print(f"✓ No existing ontology found — proceeding with creation.\n")
print(f"Creating ontology '{ontology_name}' in workspace...")

# Create the ontology
result = create_ontology_item(
    workspace_id=workspace_id,
    access_token=access_token,
    ontology_item_name=ontology_name,
    ontology_definition=ontology_definition
)

print(f"\nResponse Status Code: {result.status_code}")

if result.status_code in [200, 201]:
    print("\n✓ Ontology created successfully!")
    print(f"  Ontology Name: {ontology_name}")
    print(f"  Entity Types: 6 (ProductionLine, Station, ProductionSchedule, MachineEvent, PartEvent, MaintenanceEvent)")
    print(f"  Relationships: 8")
    print(f"  Data Bindings: Hybrid (static lakehouse + time series eventhouse)")
else:
    print(f"\n❌ Ontology creation failed with status {result.status_code}")
    print(f"\nError details:")
    try:
        print(result.json())
    except:
        print(result.text)

## Step 7: Verify GraphModel Creation

The ontology automatically creates an underlying Graph item. Check that it populated correctly.

In [ ]:
import time

print("\nVerifying GraphModel creation...")

# Wait for GraphModel to be created (may take a few seconds)
time.sleep(15)

# Refresh access token and headers
access_token = notebookutils.credentials.getToken("pbi")
headers = {
    "Authorization": f"Bearer {access_token}",
    "Accept": "application/json",
    "Content-Type": "application/json",
}

# List items again to find the auto-created GraphModel
items_resp = requests.get(
    f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items",
    headers=headers,
)

if items_resp.status_code == 200:
    items = items_resp.json().get("value", [])
    graph_items = [i for i in items if i.get("type") == "GraphModel" and ontology_name in i.get("displayName", "")]
    
    if graph_items:
        graph_model_id = graph_items[0]["id"]
        graph_model_name = graph_items[0]["displayName"]
        print(f"✓ GraphModel found: {graph_model_name}")
        print(f"  ID: {graph_model_id}")
        
        # Poll for GraphModel definition (may take time to provision)
        print("\nWaiting for GraphModel definition to become available...")
        for attempt in range(6):
            graph_def_resp = requests.get(
                f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items/{graph_model_id}/getDefinition",
                headers=headers,
            )
            
            if graph_def_resp.status_code == 200:
                graph_def = graph_def_resp.json()
                # Navigate the definition structure
                parts = graph_def.get("definition", {}).get("parts", [])
                print(f"  Definition parts: {len(parts)}")
                for part in parts:
                    print(f"    - {part.get('path', 'unknown')}")
                print(f"\n✓ GraphModel definition retrieved successfully on attempt {attempt + 1}")
                break
            elif graph_def_resp.status_code == 202:
                # Long-running operation - check location header
                print(f"  Attempt {attempt + 1}: provisioning in progress (202)...")
                time.sleep(15)
            elif graph_def_resp.status_code == 404:
                print(f"  Attempt {attempt + 1}: not ready yet (404), waiting 15s...")
                time.sleep(15)
            else:
                print(f"  Attempt {attempt + 1}: status {graph_def_resp.status_code}")
                try:
                    print(f"    {graph_def_resp.json()}")
                except:
                    print(f"    {graph_def_resp.text}")
                time.sleep(15)
        else:
            print("\n⚠️ GraphModel definition not available after retries.")
            print("  This is normal — Fabric provisions it asynchronously.")
            print("  Open the ontology in Fabric UI to check if the graph populated.")
    else:
        print("⚠️ GraphModel not found yet. Check the workspace in a few minutes.")
else:
    print(f"⚠️ Failed to list items: {items_resp.status_code}")

## Step 7b: Diagnose Empty Graph (0 nodes / 0 edges)

If the graph shows 0 nodeTypes and 0 edgeTypes, run this cell to compare declared CSV types vs actual table column types. **Any mismatch silently drops the entire entity type.**

In [ ]:
import csv

# Type mapping: Spark/SQL types -> Ontology CSV types
SPARK_TO_ONTOLOGY = {
    "string": "String", "varchar": "String",
    "bigint": "BigInt", "long": "BigInt", "int": "BigInt", "integer": "BigInt", "smallint": "BigInt", "tinyint": "BigInt",
    "double": "Double", "float": "Double", "decimal": "Double",
    "boolean": "Boolean",
    "timestamp": "DateTime", "date": "DateTime", "timestamp_ntz": "DateTime",
}

def load_csv(path):
    with open(path, 'r') as f:
        return list(csv.DictReader(f))

entity_rows = load_csv(f"{ontology_base_path}/definition/entity_types.csv")
binding_rows = load_csv(f"{ontology_base_path}/binding/binding_entity_types.csv")

# Get declared types per entity.property
declared_types = {}
for r in entity_rows:
    declared_types[(r["EntityTypeName"], r["PropertyName"])] = r["PropertyDataType"]

# Get unique NonTimeSeries table bindings (these are the lakehouse tables Fabric checks)
tables_to_check = set()
for r in binding_rows:
    if r["DataBindingType"] == "NonTimeSeries" and r["SourceType"] == "LakehouseTable":
        tables_to_check.add(r["SourceTableName"])

# Also check TimeSeries/KustoTable bindings
kusto_tables = set()
for r in binding_rows:
    if r["DataBindingType"] == "TimeSeries" and r["SourceType"] == "KustoTable":
        kusto_tables.add(r["SourceTableName"])

print("=" * 70)
print("DIAGNOSING EMPTY GRAPH: Checking table schemas vs CSV declarations")
print("=" * 70)

# Check lakehouse tables (NonTimeSeries bindings)
print("\n--- Lakehouse Tables (NonTimeSeries bindings via dbo schema) ---\n")
mismatches = []
for table_name in sorted(tables_to_check):
    try:
        df = spark.sql(f"DESCRIBE dbo.{table_name}")
        actual_cols = {row["col_name"]: row["data_type"] for row in df.collect() if not row["col_name"].startswith("#")}
        
        # Find all binding rows for this table
        table_bindings = [r for r in binding_rows if r["SourceTableName"] == table_name and r["DataBindingType"] == "NonTimeSeries"]
        
        print(f"  {table_name}: {len(actual_cols)} columns, {len(table_bindings)} bindings")
        
        for b in table_bindings:
            col = b["BindingSourceColumnName"]
            declared = b["PropertyDataType"]
            actual = actual_cols.get(col)
            
            if actual is None:
                print(f"    ❌ {col}: COLUMN NOT FOUND in table!")
                mismatches.append(f"{table_name}.{col}: column missing")
            else:
                expected = SPARK_TO_ONTOLOGY.get(actual.lower(), f"UNKNOWN({actual})")
                if expected != declared:
                    print(f"    ❌ {col}: declared={declared}, actual={actual} (should be {expected})")
                    mismatches.append(f"{table_name}.{col}: {declared} != {expected} (actual: {actual})")
                else:
                    print(f"    ✓ {col}: {declared} matches {actual}")
    except Exception as e:
        print(f"  ❌ {table_name}: TABLE NOT FOUND - {e}")
        mismatches.append(f"{table_name}: table does not exist in dbo schema")

# Check KQL tables exist (TimeSeries bindings)
print(f"\n--- KQL Tables (TimeSeries bindings) ---\n")
print(f"  Tables referenced: {', '.join(sorted(kusto_tables))}")
print(f"  (KQL table types are checked by the Eventhouse, not the lakehouse)")
print(f"  Verify these tables exist in ManufacturingEH with OneLake availability enabled")

# Summary
print("\n" + "=" * 70)
if mismatches:
    print(f"❌ {len(mismatches)} MISMATCH(ES) FOUND - these cause the empty graph:\n")
    for m in mismatches:
        print(f"  {m}")
    print(f"\nFix the PropertyDataType in entity_types.csv and binding_entity_types.csv to match.")
else:
    print("✓ All declared types match actual table columns.")
    print("  If graph is still empty, check:")
    print("  1. OneLake availability is enabled on ManufacturingEH KQL DB")
    print("  2. KQL tables have data")
    print("  3. Tables are accessible via dbo schema in the lakehouse")
    print("  4. Wait 2-3 minutes and refresh the ontology page")

## Next Steps

1. **Verify OneLake availability** is enabled on the ManufacturingEH KQL Database
2. **Open the ontology** in Fabric UI to view the entity types and relationships
3. **Preview the graph** to ensure data bindings are working (should show 6 node types, 8 edge types)
4. **Create a Data Agent** connected to this ontology for natural language querying

### Example Data Agent Queries
- "List all production lines with their station counts"
- "Show me all stations on Line-A with their machine types"
- "Which stations have the highest fault rates this week?"
- "Show all maintenance events for Line-C station 2 in the last 24 hours"
- "What is the history of part P-00042?"
- "Which machines have had the most maintenance work orders?"
- "Show me all open maintenance work orders"
- "What's the buffer status for stations on Line-B?" (query MachineEvents table directly)

**Note**: OEE calculations are not in the ontology (OEE_5min is a materialized view). Query the `OEE_5min` KQL table directly or compute OEE from raw MachineEvent data.